# Episode 9: Choosing the Right Pattern

You've now learned five orchestration patterns. The hard part isn't learning them — it's knowing which one to pick.

**By the end of this episode, you will:**
- Have a decision framework for selecting the right pattern
- Understand the trade-offs across control, cost, flexibility, and debuggability
- Know how AG2 compares to other multi-agent frameworks

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## The patterns so far

Over Episodes 5-8, you've built with five distinct orchestration patterns. Each one solves a different problem, and choosing the wrong one leads to poor results, wasted tokens, or brittle systems. A Tufts University study found that a GPT-3.5 multi-agent system outperformed a single GPT-4 agent on complex tasks — but only when the orchestration pattern matched the problem structure.

This episode gives you a framework for making that match.

## Pattern comparison

Here's the landscape at a glance. Scan this table for the broad strokes, then we'll dig into how to choose.

| Pattern | Speaker Selection | Control Level | Cost | Best For |
|---------|------------------|--------------|------|----------|
| **RoundRobin** | Fixed order | High | Low | Brainstorming, reviews |
| **Auto** | LLM decides | Low | High (extra LLM call) | Open-ended tasks |
| **Default (Handoffs)** | Explicit rules | Highest | Low | Support routing, compliance |
| **Pipeline** | Sequential chain | High | Low | Data processing, approvals |
| **Hierarchical** | Manager delegates | Medium | Medium | Reports, complex analysis |
| **Redundant** | All work in parallel | Low | Highest (N x cost) | Validation, high-stakes |

## Decision flowchart

Here's how I think about it:

```
Is the agent order predetermined?
  YES --> RoundRobin
  NO  |
      v
Do you have explicit routing rules?
  YES --> Default (Handoffs)
  NO  |
      v
Is it a linear pipeline?
  YES --> Pipeline (Sequential)
  NO  |
      v
Does one agent delegate to sub-teams?
  YES --> Hierarchical
  NO  |
      v
Do you need diverse perspectives on same task?
  YES --> Redundant
  NO  |
      v
Auto (let LLM decide)
```

Notice that Auto is the *last* option, not the first. That's intentional. The more structure you can give your system, the more predictable and debuggable it becomes. Reach for Auto when you genuinely can't predict the flow — not as a shortcut to avoid thinking about routing.

## Exercise: Which pattern would you choose?

These aren't trick questions, but the answers might surprise you. For each scenario, pick a pattern before revealing the answer.

### Scenario 1

**"A code review system where Author, Reviewer, and QA each comment in turn."**

Think about it...

<details>
<summary>Click to reveal answer</summary>

**RoundRobin** -- The order is fixed and predictable. Each role contributes on every cycle. There's no need for dynamic routing.

```python
pattern = RoundRobinPattern(
    initial_agent=author,
    agents=[author, reviewer, qa],
)
```
</details>

### Scenario 2

**"A customer support system routing billing, technical, and general queries."**

Think about it...

<details>
<summary>Click to reveal answer</summary>

**Default (Handoffs)** -- You have clear categories and routing rules. The triage agent evaluates conditions and hands off to the right specialist.

```python
triage_agent = ConversableAgent(
    name="triage",
    handoffs=[
        OnCondition(target=AgentTarget(billing_agent), condition=StringLLMCondition(...)),
        OnCondition(target=AgentTarget(tech_agent), condition=StringLLMCondition(...)),
        OnCondition(target=AgentTarget(general_agent), condition=StringLLMCondition(...)),
    ],
    ...
)
```
</details>

### Scenario 3

**"A content pipeline: research -> write -> edit -> publish."**

Think about it...

<details>
<summary>Click to reveal answer</summary>

**Pipeline (Sequential)** -- Each stage transforms the output and passes it to the next. No branching or delegation needed.

```python
research_agent.set_after_work(AgentTarget(write_agent))
write_agent.set_after_work(AgentTarget(edit_agent))
edit_agent.set_after_work(AgentTarget(publish_agent))
publish_agent.set_after_work(TerminateTarget())
```
</details>

## Additional Content

### How AG2 compares to other frameworks

Let's be honest about the landscape. AG2 isn't the only multi-agent framework, and understanding the alternatives helps you make informed choices.

| Framework | Approach | AG2 Equivalent |
|-----------|----------|----------------|
| **CrewAI** | Sequential / Hierarchical Process | Pipeline / Hierarchical |
| **LangGraph** | State machine with nodes + edges | DefaultPattern with handoffs |
| **Google ADK** | Agent-to-agent routing | DefaultPattern + A2A protocol |

Each framework has real strengths. CrewAI offers a clean high-level API. LangGraph gives you fine-grained state machine control. Google ADK integrates tightly with Google's ecosystem.

AG2's differentiator is that all patterns live in one framework. You can switch between RoundRobin, Auto, and Default by changing one line — the agents stay the same. Other frameworks often lock you into a single orchestration paradigm, which becomes painful when your requirements evolve.

### Trade-offs matrix

Every pattern choice involves trade-offs across multiple dimensions. This matrix helps you weigh what matters most for your use case.

| Dimension | RoundRobin | Auto | Handoffs | Pipeline | Hierarchical |
|-----------|-----------|------|----------|----------|-------------|
| **Control** | High | Low | Highest | High | Medium |
| **Flexibility** | Low | Highest | Medium | Low | High |
| **Cost** | Predictable | Highest | Predictable | Lowest | Medium |
| **Debugging** | Easiest | Hardest | Easy | Easy | Medium |
| **Scalability** | Linear | Depends | Good | Linear | Good |

A few rules of thumb that have served me well. Start with the **simplest pattern** that could work — often RoundRobin or Pipeline. Move to **handoffs** when you need routing logic. Use **Auto** only when you genuinely can't predict the conversation flow. And reach for **Hierarchical** when tasks naturally decompose into subtasks that require synthesis.

### Quick reference: one-line pattern switch

One of AG2's strengths is that you can switch patterns without changing your agents. Here's the key part:

```python
from autogen.agentchat.group.patterns import (
    RoundRobinPattern,
    AutoPattern,
    DefaultPattern,
)

agents = [agent_a, agent_b, agent_c]

# Switch between patterns by changing one line:
pattern = RoundRobinPattern(initial_agent=agent_a, agents=agents)
# pattern = AutoPattern(initial_agent=agent_a, agents=agents, group_manager_args={...})
# pattern = DefaultPattern(initial_agent=agent_a, agents=agents)
```

This makes prototyping fast. Build your agents once, then experiment with different patterns until you find the right fit.

## What's Next

You've got the theory. But can you apply it? In the next episode, you'll build a complete system from scratch — and the first decision you'll make is which pattern to use.

In **Episode 10**, you'll apply these patterns to build a **complete customer service system** — combining routing, escalation, and context management into a production-ready multi-agent application.